### Architecture

### Imports

In [1]:
import os
import joblib
import pandas as pd

### Create Output Folder

In [2]:
os.makedirs("batch_predictions", exist_ok=True)

print("Folder created successfully.")

Folder created successfully.


### Load Production Model

In [3]:
model = joblib.load(
    "model_registry/production/model.pkl"
)

print("Production model loaded.")

Production model loaded.


### Load Preprocessor

In [4]:
preprocessor = joblib.load(
    "models/preprocessor.pkl"
)

print("Preprocessor loaded.")

Preprocessor loaded.


### Load Batch Dataset
Use the original customer CSV:

In [6]:
batch_data = pd.read_csv(
    "data/customer_data.csv"
)

batch_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Separate Features
Remove target column:

In [7]:
X_batch = batch_data.drop(
    columns=["Churn"]
)

X_batch.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [9]:
X_batch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [10]:
X_batch.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [11]:
for col in X_batch.columns:
    print(col)
    print(X_batch[col].unique()[:10])
    print()

customerID
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' '7795-CFOCW' '9237-HQITU'
 '9305-CDSKC' '1452-KIOVK' '6713-OKOMC' '7892-POOKP' '6388-TABGU']

gender
['Female' 'Male']

SeniorCitizen
[0 1]

Partner
['Yes' 'No']

Dependents
['No' 'Yes']

tenure
[ 1 34  2 45  8 22 10 28 62 13]

PhoneService
['No' 'Yes']

MultipleLines
['No phone service' 'No' 'Yes']

InternetService
['DSL' 'Fiber optic' 'No']

OnlineSecurity
['No' 'Yes' 'No internet service']

OnlineBackup
['Yes' 'No' 'No internet service']

DeviceProtection
['No' 'Yes' 'No internet service']

TechSupport
['No' 'Yes' 'No internet service']

StreamingTV
['No' 'Yes' 'No internet service']

StreamingMovies
['No' 'Yes' 'No internet service']

Contract
['Month-to-month' 'One year' 'Two year']

PaperlessBilling
['Yes' 'No']

PaymentMethod
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

MonthlyCharges
[ 29.85  56.95  53.85  42.3   70.7   99.65  89.1   29.75 104.8   56.15]

TotalCharges
['29.85' '188

### Transform Features

In [13]:
# Remove ID column
X_batch = X_batch.drop(columns=["customerID"])

# Fix TotalCharges datatype
X_batch["TotalCharges"] = pd.to_numeric(
    X_batch["TotalCharges"],
    errors="coerce"
)

X_batch["TotalCharges"] = X_batch["TotalCharges"].fillna(0)

# Transform data
X_batch_processed = preprocessor.transform(X_batch)

print("Batch preprocessing completed.")

Batch preprocessing completed.


### Generate Predictions


In [14]:
predictions = model.predict(
    X_batch_processed
)

probabilities = model.predict_proba(
    X_batch_processed
)[:, 1]

print("Batch predictions completed.")

Batch predictions completed.


### Add Predictions to Dataset

In [15]:
batch_data["Prediction"] = predictions

batch_data["Churn_Probability"] = probabilities

batch_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Prediction,Churn_Probability
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1,0.622651
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.5,No,0,0.044294
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0,0.286630
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,0.028434
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,0.690068


### Convert Prediction Labels

In [16]:
batch_data["Prediction_Label"] = (
    batch_data["Prediction"]
    .map({
        0: "No Churn",
        1: "Churn"
    })
)

batch_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Prediction,Churn_Probability,Prediction_Label
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1,0.622651,Churn
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.5,No,0,0.044294,No Churn
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0,0.286630,No Churn
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,0.028434,No Churn
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,0.690068,Churn


### Save Prediction Report


In [17]:
batch_data.to_csv(
    "batch_predictions/customer_predictions.csv",
    index=False
)

print("Prediction report saved successfully.")

Prediction report saved successfully.


### Business Summary

In [18]:
summary = {

    "Total Customers":
        len(batch_data),

    "Predicted Churn Customers":
        (batch_data["Prediction"] == 1).sum(),

    "Predicted Non-Churn Customers":
        (batch_data["Prediction"] == 0).sum()
}

summary

{'Total Customers': 7043,
 'Predicted Churn Customers': np.int64(1580),
 'Predicted Non-Churn Customers': np.int64(5463)}

### High-Risk Customers
Customers with probability above 80%:

In [19]:
high_risk_customers = batch_data[
    batch_data["Churn_Probability"] >= 0.80
]

high_risk_customers.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Prediction,Churn_Probability,Prediction_Label
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,0.802203,Churn
301,8098-LLAZX,Female,1,No,No,4,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,95.45,396.1,Yes,1,0.842559,Churn
582,2865-TCHJW,Female,1,No,No,4,Yes,Yes,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,89.20,346.2,Yes,1,0.815600,Churn
585,5192-EBGOV,Female,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,85.70,85.7,Yes,1,0.824909,Churn
905,0781-LKXBR,Male,1,No,No,9,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,100.50,918.6,Yes,1,0.821453,Churn


### Save High-Risk Report

In [20]:
high_risk_customers.to_csv(
    "batch_predictions/high_risk_customers.csv",
    index=False
)

print("High-risk customer report saved.")

High-risk customer report saved.


### Top 10 Highest Risk Customers

In [21]:
top_customers = (
    batch_data
    .sort_values(
        by="Churn_Probability",
        ascending=False
    )
    .head(10)
)

top_customers

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Prediction,Churn_Probability,Prediction_Label
1976,9497-QCMMS,Male,1,No,No,1,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,93.55,93.55,Yes,1,0.854098,Churn
4800,9300-AGZNL,Male,1,No,No,1,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,94.00,94,Yes,1,0.853103,Churn
1410,7024-OHCCK,Female,1,No,No,2,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,93.85,170.85,Yes,1,0.852281,Churn
3749,4424-TKOPW,Male,1,No,No,2,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,93.85,196.75,Yes,1,0.850156,Churn
2208,7216-EWTRS,Female,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,100.80,100.8,Yes,1,0.849348,Churn
6368,2720-WGKHP,Male,1,No,No,2,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,94.00,181.7,Yes,1,0.849268,Churn
3380,5178-LMXOP,Male,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,95.10,95.1,Yes,1,0.847466,Churn
997,1374-DMZUI,Female,1,No,No,4,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,94.30,424.45,Yes,1,0.846328,Churn
3159,5150-ITWWB,Male,1,No,No,3,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,94.85,335.75,No,1,0.845807,Churn
301,8098-LLAZX,Female,1,No,No,4,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,95.45,396.1,Yes,1,0.842559,Churn


### Pipeline Completion Message

In [22]:
print("""
Batch Inference Pipeline Completed

✓ Model Loaded
✓ Data Loaded
✓ Features Processed
✓ Batch Predictions Generated
✓ Prediction Probabilities Generated
✓ CSV Reports Saved
✓ High-Risk Customers Identified

Enterprise Batch Scoring Ready
""")


Batch Inference Pipeline Completed

✓ Model Loaded
✓ Data Loaded
✓ Features Processed
✓ Batch Predictions Generated
✓ Prediction Probabilities Generated
✓ CSV Reports Saved
✓ High-Risk Customers Identified

Enterprise Batch Scoring Ready

